# TP5 — Séries temporelles : prévoir la demande de vélos en libre-service (Énoncé)

Cours associé : [TP5_Cours.md](TP5_Cours.md) · Pense-bête : [Pense_bete_Pandas.md](../Pense_bete_Pandas.md)

## Contexte

Capital Bikeshare, l'opérateur de vélos en libre-service de Washington DC, doit chaque soir décider combien de vélos redistribuer et combien d'agents mobiliser le lendemain. Il vous demande un modèle qui **prévoit le nombre total de locations du lendemain**.

Le fichier `data/bike_sharing_day.csv` contient le nombre de locations par jour en 2011 et 2012, avec des informations de calendrier et de météo :

| Colonne | Signification |
|---|---|
| `dteday` | date |
| `holiday`, `workingday` | jour férié ; jour ouvré (ni férié, ni week-end) |
| `weathersit` | météo : 1 = dégagé, 2 = brume / nuageux, 3 = pluie ou neige légère, 4 = intempéries |
| `temp`, `atemp` | température et température ressentie, normalisées (÷ 41 et ÷ 50 °C) |
| `hum`, `windspeed` | humidité et vent, normalisés |
| `casual`, `registered` | locations par des usagers occasionnels / abonnés |
| `cnt` | **nombre total de locations** (cible) |

## Déroulé

| Partie | Contenu |
|---|---|
| 1 | Exploration et décomposition de la série |
| 2 | Découpage temporel et références naïves |
| 3 | Modèle ARIMA |
| 4 | Approche supervisée avec XGBoost |
| 5 | Synthèse |
| 6 | *Facultatif* : approfondissements |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.figsize"] = (12, 4)
RANDOM_STATE = 42

## Partie 1 — Exploration et décomposition de la série

**1.1** Chargez les données dans `df` en convertissant `dteday` en date et en l'utilisant comme index. Vérifiez qu'il n'y a ni date manquante ni doublon : l'index doit couvrir tous les jours du 1ᵉʳ janvier 2011 au 31 décembre 2012.

*Indice : `pd.date_range(debut, fin, freq="D")` construit la liste complète des jours.*

In [ ]:
# TODO : chargement avec index de dates et contrôles


**1.2** Tracez la série `cnt`, avec par-dessus sa moyenne mobile sur 30 jours. Décrivez ce que vous voyez (tendance, saisonnalité, valeurs extrêmes).

In [ ]:
# TODO : série et moyenne mobile


**1.3** Affichez les 5 jours avec le moins de locations. Que s'est-il passé ? (Cherchez les dates si besoin.)

In [ ]:
# TODO : jours extrêmes


**1.4** Calculez le nombre moyen de locations par **jour de la semaine** et par **mois**, en distinguant usagers occasionnels (`casual`) et abonnés (`registered`). Les deux populations se comportent-elles de la même façon ?

In [ ]:
# TODO : profils hebdomadaire et mensuel


✍️ **Votre réponse :**

…

**1.5** Décomposez la série `y` avec `seasonal_decompose` (modèle additif, période 7). Qu'est-ce que la composante « tendance » capture ici ? Quelle est l'amplitude de la saisonnalité hebdomadaire ?

In [ ]:
# TODO : décomposition


✍️ **Votre réponse :**

…

## Partie 2 — Découpage temporel et références naïves

On évalue les modèles sur les **trois derniers mois** (octobre à décembre 2012), en **prévision à un pas** : chaque soir, on prévoit la journée du lendemain en connaissant tout le passé.

**2.1** Créez `y_train` (jusqu'au 30 septembre 2012 inclus) et `y_test` (à partir du 1ᵉʳ octobre 2012). Tracez-les avec deux couleurs.

In [ ]:
DATE_COUPURE = "2012-10-01"
# TODO : découpage temporel

La fonction suivante mémorise les métriques de chaque méthode sur le jeu de test : MAE, RMSE, erreur relative médiane et **biais** (erreur moyenne ; positif si le modèle sous-estime).

In [ ]:
resultats = {}

def evaluer(nom, prevision):
    prevision = prevision.loc[y_test.index]
    resultats[nom] = {
        "MAE": mean_absolute_error(y_test, prevision),
        "RMSE": root_mean_squared_error(y_test, prevision),
        "MAPE (%)": (np.abs(y_test - prevision) / y_test).median() * 100,   # médiane : robuste au jour de l'ouragan
        "biais": (y_test - prevision).mean(),   # > 0 : le modèle sous-estime en moyenne
    }
    return pd.DataFrame(resultats).T.round(1)

**2.2** Calculez et évaluez trois prévisions naïves à un pas sur la période de test :

- **persistance** : la valeur de la veille ;
- **naïve saisonnière** : la valeur du même jour la semaine précédente ;
- **moyenne des 7 derniers jours**.

*Indice : ces prévisions se calculent sur la série complète `y` avec `shift` et `rolling`, puis on ne garde que la période de test.*

In [ ]:
# TODO : prévisions naïves


✍️ **Votre réponse :**

…

## Partie 3 — Modèle ARIMA

**3.1** Ajustez un modèle ARIMA(1, 1, 1) sur `y_train` : `SARIMAX(y_train, order=(1, 1, 1))`, puis `.fit(disp=False)` ; gardez le résultat dans `res`. Affichez le tableau des coefficients (`res.summary().tables[1]`). Les coefficients sont-ils significatifs (colonne `P>|z|`) ?

*Le choix des ordres (1, 1, 1) est justifié dans la partie facultative (stationnarité, autocorrélation, comparaison avec un modèle saisonnier).*

In [ ]:
# TODO : ajustement ARIMA


✍️ **Votre réponse :**

…

**3.2** Prévision **multi-pas** : prévoyez d'un coup les 92 jours de la période de test (`res.get_forecast(steps=len(y_test))`) et tracez la prévision avec son intervalle de confiance à 95 % par-dessus les vraies valeurs. Évaluez-la avec la fonction `evaluer`. Qu'observez-vous ?

In [ ]:
# TODO : prévision multi-pas


✍️ **Votre réponse :**

…

**3.3** Prévision **à un pas** : on garde les paramètres appris sur l'entraînement, et on fournit au modèle les vraies valeurs au fur et à mesure. Utilisez `res.append(y_test)` pour obtenir un modèle qui connaît toute la série (sans réajuster les paramètres), puis `predict(start=DATE_COUPURE)` qui renvoie, pour chaque jour, la prévision faite la veille. Évaluez et tracez.

In [ ]:
# TODO : prévision à un pas


✍️ **Votre réponse :**

…

## Partie 4 — Approche supervisée avec XGBoost

On transforme la série en tableau de régression : une ligne par jour, des variables construites à partir du passé, du calendrier et de la météo.

**4.1** Construisez un DataFrame `feat` contenant :

- `lag_1`, `lag_2`, `lag_7` : les locations 1, 2 et 7 jours avant ;
- `moy_7j` : la moyenne des locations des 7 jours **précédents** (attention à la fuite de données) ;
- `jour_semaine`, `mois` : issus de l'index ;
- `holiday`, `workingday`, `weathersit`, `temp`, `hum`, `windspeed` : repris de `df` ;
- la cible `cnt`.

Supprimez les lignes contenant des valeurs manquantes (les 7 premiers jours).

In [ ]:
# TODO : construction des variables


**4.2** Pourquoi faut-il écrire `y.shift(1).rolling(7).mean()` et non `y.rolling(7).mean()` ? Et pourquoi l'utilisation de la météo **du jour même** est-elle discutable ?

✍️ **Votre réponse :**

…

**4.3** Découpez `feat` en `X_train`, `y_train_sup`, `X_test`, `y_test_sup` selon `DATE_COUPURE`. Entraînez un `XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=3, subsample=0.8)` sur le jeu d'entraînement, prévoyez la période de test, évaluez la prévision avec `evaluer` et tracez-la. Tracez aussi l'importance des variables.

In [ ]:
# TODO : découpage, entraînement, évaluation, importances


**4.4** Pour mesurer l'apport de la météo, réentraînez le même modèle **sans** les variables météo (`weathersit`, `temp`, `hum`, `windspeed`) et évaluez-le.

In [ ]:
# TODO : modèle sans météo


✍️ **Votre réponse :**

…

## Partie 5 — Synthèse

**5.1** Affichez le tableau `resultats` trié par MAE.

In [ ]:
# TODO : tableau récapitulatif


**5.2** Rédigez une recommandation pour l'opérateur : quel modèle, quelle précision, quelles données collecter, quelles limites ?

✍️ **Votre réponse :**

…

## Partie 6 — *Facultatif* : approfondissements

*À faire si vous avez terminé le reste.* Les questions 6.1 à 6.3 justifient le choix du modèle ARIMA(1, 1, 1) de la partie 3 ; les questions 6.4 et 6.5 portent sur la validation croisée d'une série temporelle.

**6.1** Appliquez le test ADF à `y`, puis à sa différence d'ordre 1 (`y.diff()`). Concluez sur la stationnarité. Tracez la série différenciée.

In [ ]:
# TODO : test ADF


✍️ **Votre réponse :**

…

**6.2** Tracez l'ACF et la PACF (40 décalages) de la série **différenciée**. Quels décalages sont significatifs ?

In [ ]:
# TODO : ACF et PACF


✍️ **Votre réponse :**

…

**6.3** Une partie saisonnière de période 7 améliore-t-elle le modèle ? Ajustez `SARIMAX(y_train, order=(1, 1, 1), seasonal_order=(1, 0, 1, 7))`, affichez ses coefficients et comparez son AIC à celui du modèle `res` de la partie 3 (plus l'AIC est faible, mieux c'est).

In [ ]:
# TODO : SARIMA et comparaison des AIC


✍️ **Votre réponse :**

…

**6.4** Évaluez le modèle XGBoost de la partie 4 par **validation croisée temporelle** sur le jeu d'entraînement (`TimeSeriesSplit(n_splits=5)`, score MAE). Affichez la MAE de chaque pli.

In [ ]:
# TODO : validation croisée temporelle


**6.5** Les MAE des plis sont très différentes. Affichez les périodes de validation de chaque pli (`tscv.split(X_train)`) pour comprendre pourquoi.

In [ ]:
# TODO : périodes des plis


✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

- Ajoutez la météo comme variable **exogène** au modèle SARIMA (`SARIMAX(y_train, exog=..., ...)`) : combine-t-il les avantages des deux approches ?
- Prévoyez séparément `casual` et `registered` avec XGBoost, puis additionnez les prévisions : est-ce meilleur que de prévoir `cnt` directement ?
- Comparez quelques ordres de SARIMA à l'aide de l'AIC.
- Un arbre ne sait pas extrapoler (TP4) : il peine quand le niveau de la série dépasse ce qu'il a vu. Faites prédire à XGBoost le **rapport** `cnt / moy_7j` plutôt que `cnt`, puis remultipliez par `moy_7j` : l'erreur diminue-t-elle ?

In [ ]:
# Bonus : SARIMAX avec la météo en variables exogènes


In [ ]:
# Bonus : prédire le rapport à la moyenne des 7 derniers jours
